# Exercise 5: structured extraction

20 abstracts of clinical studies. The goal: a table that a program can use.

1. One abstract: a prompt, JSON mode, a dict
2. Check the fields and their types with `pydantic`
3. The 20 abstracts: a table and a CSV file

Option, if you finish early: a strict schema, or a `confidence` field.

Run the cells one by one, from the top: **Shift+Enter**. After you change a cell, run it again, then the cells below it.

In [ ]:
import json
from typing import Literal

import pandas as pd
from IPython.display import Markdown
from pydantic import BaseModel, ValidationError

import checks
from course import OUTPUT, abstracts, ask_json, write_csv

pd.set_option("display.max_colwidth", 120)
corpus = {item["id"]: item["text"] for item in abstracts()}

## 1 · One abstract

> **The idea:** a program needs fixed fields, not sentences, so we ask for JSON (slide "Un programme lit des champs, pas des phrases")

**Complete the prompt**: one field of the list is missing. Use the allowed values below.

`study_design`: randomised controlled trial, cluster randomised trial, crossover trial, secondary analysis of a trial, cross-sectional study · `effect_found`: yes, no, mixed, not applicable

In [ ]:
fields = """Extract these fields from the abstract, as a JSON object:
- population: who was studied (text)
- intervention: what was tested, and against what (text)
- sample_size: number of participants (integer)
- effect_found: a significant benefit on the primary outcome? one of: yes, no, mixed, not applicable
- main_result: the main result, in one sentence (text)"""

In [ ]:
checks.ex5_fields(fields)

**Choose an abstract**, `"A01"` to `"A20"`, then run the two cells.

In [ ]:
abstract_id = "A01"

In [ ]:
raw, seconds = ask_json("You extract structured data from abstracts of clinical studies.", f"{fields}\n\n<abstract>\n{corpus[abstract_id]}\n</abstract>")
print(f"{seconds:.1f} s")
print(json.dumps(json.loads(raw), indent=2, ensure_ascii=False))

- JSON mode: the API returns valid JSON
- It does not promise your fields, or the right values

<details><summary>Question</summary>

Compare with the abstract (`print(corpus[abstract_id])`). Is each value correct?

</details>

## 2 · Check with pydantic

> **The idea:** valid JSON can hold a value of the wrong type. A pydantic schema rejects it (slide "`pydantic` : un formulaire avec des champs typés")

**Write one line**: the field `sample_size`, an integer.

In [ ]:
class Extraction(BaseModel):
    population: str
    intervention: str
    ...  # TODO
    study_design: Literal["randomised controlled trial", "cluster randomised trial", "crossover trial", "secondary analysis of a trial", "cross-sectional study"]
    effect_found: Literal["yes", "no", "mixed", "not applicable"]
    main_result: str

In [ ]:
checks.ex5_model(Extraction)

In [ ]:
bad = '{"population": "adults", "intervention": "metformin", "sample_size": "about 250", "study_design": "trial", "effect_found": "yes", "main_result": "..."}'
try:
    Extraction.model_validate_json(bad)
    verdict = "accepted"
except ValidationError as error:
    verdict = f"rejected:\n\n```\n{error}\n```"
Markdown(f"A bad answer is {verdict}")

- The check rejects a wrong type or a value not in the list, and gives the reason
- Valid JSON with the wrong number is still wrong: "87,449 women" can become 87

<details><summary>Question</summary>

Does your extraction from part 1 pass the check? Try `Extraction.model_validate_json(raw)`.

</details>

## 3 · The 20 abstracts

> **The idea:** the check finds badly formed answers, not wrong values (slide "JSON valide, valeur fausse")

20 calls, one after the other. The free tier allows 8,000 tokens per minute, so some calls wait and try again (HTTP 429). This takes about 2 minutes.

In [ ]:
rows = []
for item_id, text in corpus.items():
    print(item_id, end=" ")
    answer, _ = ask_json("You extract structured data from abstracts of clinical studies.", f"{fields}\n\n<abstract>\n{text}\n</abstract>")
    try:
        rows.append({"id": item_id, **Extraction.model_validate_json(answer).model_dump(), "error": ""})
    except ValidationError as error:
        rows.append({"id": item_id, "error": str(error).splitlines()[0]})
write_csv(OUTPUT / "extraction.csv", rows)
display(Markdown(f"Saved to `data/output/extraction.csv`: {len(rows)} rows"))
pd.DataFrame(rows)

**Key points**

- A table that a program can use, in 2 minutes
- Ask in the prompt < JSON mode < validation
- Some values are wrong: how do you find them without reading all 20? That is exercise 6

<details><summary>Question</summary>

Look at `sample_size`. Which values look wrong?

</details>

---
## Option

- **Strict schema**: Groq accepts `response_format={"type": "json_schema", ...}` with the schema from `Extraction.model_json_schema()`. The API then returns your fields
- **Confidence**: add a `confidence` field (0 to 1) to the prompt and to `Extraction`, then run again. In exercise 6, does the confidence predict the errors?